<div align="center">
  <h2>Міністерство освіти і науки України</h2>
  <h2>Національний технічний університет України</h2>
  <h2>«Київський політехнічний інститут ім. Ігоря Сікорського»</h2>
  <h2>Факультет інформатики та обчислювальної техніки</h2>
  <h2>Кафедра обчислювальної техніки</h2>
  <br>
</div>

<div align="right">
    <br>
    <br>
<center>
<h2>Комп'ютерний практикум №1</h2>
    <h2>з дисципліни</h2>
    <h2>«Штучний інтелект в задачах обробки зображень»</h2>
    <h2>на тему:</h2>
    <h2>«Розпізнавання номерних знаків за допомогою Tesseract»</h2>
</center>
    <br>
    <br>
    <br>
    <br>
    <br>
    <br>
    <br>
    <br>
Виконали: <br>
Студенти ІІІ курсу ФІОТ <br>
групи ІО-34 <br>
Токарюк С. Б., Рибачок М. В.<br>
</div>

<center>
Київ - 2026
</center>



## 1. Мета роботи
Навчитися розпізнавати жести руки та на їх основі контролювати комп’ютер, в даному випадку звук.

## 3. Теоретичні відомості

### OpenCV
**OpenCV** (Open Source Computer Vision Library) — відкрита бібліотека комп'ютерного зору. У цій роботі використовується для:
- захоплення відео з вебкамери
- обробки та відображення зображень
- малювання кіл, ліній та тексту на кадрі

### MediaPipe + cvzone
**MediaPipe** — фреймворк від Google для розпізнавання рук у реальному часі. У версії 0.10+ змінився API (більше немає `mp.solutions`).
**cvzone** — Python-бібліотека, яка є зручною обгорткою над mediapipe 0.10+ і надає простий клас `HandDetector` з методами:
- `findHands()` — знаходить руки на зображенні та повертає landmark-и
- `fingersUp()` — повертає список з 5 елементів (1 = палець піднятий, 0 = опущений)
- `findDistance()` — відстань між двома точками

### pulsectl (Linux)
**pulsectl** — Python-бібліотека для керування PulseAudio / PipeWire на Linux:
- отримання списку аудіо-пристроїв
- встановлення рівня гучності
- вмикання/вимикання мікрофону

---
# 4. Виконання роботи
### 1. Встановлення бібліотек

In [ ]:
# Встановлення бібліотек (mediapipe 0.10+ — без cvzone)
%pip install opencv-python mediapipe pulsectl numpy


### 2. Імпорт бібліотек та сумісний HandDetector


In [ ]:
import cv2
import math
import numpy as np
import pulsectl
import urllib.request
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions
from mediapipe.tasks.python.core.base_options import BaseOptions
from mediapipe.tasks.python.vision.core.vision_task_running_mode import VisionTaskRunningMode

# ── Завантаження моделі (один раз, ~8 МБ) ────────────────────────────────
_MODEL_PATH = Path.home() / '.mediapipe' / 'hand_landmarker.task'
_MODEL_URL  = ('https://storage.googleapis.com/mediapipe-models/'
               'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task')

def _ensure_model() -> str:
    _MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not _MODEL_PATH.exists():
        print('Завантаження моделі hand_landmarker.task (~8 МБ)...')
        urllib.request.urlretrieve(_MODEL_URL, _MODEL_PATH)
        print(f'Збережено: {_MODEL_PATH}')
    return str(_MODEL_PATH)

# ── HandDetector — сумісний інтерфейс з cvzone ───────────────────────────
class HandDetector:
    """Замінює cvzone.HandTrackingModule.HandDetector для mediapipe 0.10+.
    Інтерфейс збережено: findHands(), fingersUp(), findDistance().
    """
    _TIP_IDS = [4, 8, 12, 16, 20]

    def __init__(self, staticMode=False, maxHands=2, modelComplexity=1,
                 detectionCon=0.5, minTrackCon=0.5):
        opts = HandLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=_ensure_model()),
            running_mode=VisionTaskRunningMode.IMAGE,
            num_hands=maxHands,
            min_hand_detection_confidence=float(detectionCon),
            min_hand_presence_confidence=float(detectionCon),
            min_tracking_confidence=float(minTrackCon),
        )
        self._det = HandLandmarker.create_from_options(opts)

    def findHands(self, img, draw=True, flipType=True):
        """Повертає (list[hand_dict], img). hand_dict має ключі: lmList, bbox, center, type."""
        h, w = img.shape[:2]
        rgb    = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        res    = self._det.detect(mp_img)

        all_hands = []
        for idx, hand_lms in enumerate(res.hand_landmarks):
            lm = [[int(p.x * w), int(p.y * h), p.z] for p in hand_lms]

            xs, ys = [p[0] for p in lm], [p[1] for p in lm]
            pad = 20
            bbox = (max(0, min(xs)-pad), max(0, min(ys)-pad),
                    max(xs)-min(xs)+2*pad, max(ys)-min(ys)+2*pad)

            label = 'Right'
            if idx < len(res.handedness):
                raw = res.handedness[idx][0].category_name  # 'Left'/'Right' у дзеркалі
                label = ('Left' if raw == 'Right' else 'Right') if flipType else raw

            all_hands.append({
                'lmList': lm,
                'bbox':   bbox,
                'center': ((min(xs)+max(xs))//2, (min(ys)+max(ys))//2),
                'type':   label,
            })

            if draw:
                _CONNECTIONS = [
                    (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
                    (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
                    (0,17),(17,18),(18,19),(19,20),(5,9),(9,13),(13,17)
                ]
                for a, b in _CONNECTIONS:
                    cv2.line(img, (lm[a][0], lm[a][1]), (lm[b][0], lm[b][1]),
                             (0, 255, 0), 2)
                for p in lm:
                    cv2.circle(img, (p[0], p[1]), 5, (255, 0, 255), cv2.FILLED)

        return all_hands, img

    def fingersUp(self, myHand):
        """Повертає [1/0]*5 (великий, вказівний, середній, безіменний, мізинець)."""
        lm, hand_type = myHand['lmList'], myHand['type']
        t = self._TIP_IDS
        thumb_up = (lm[t[0]][0] > lm[t[0]-1][0]) if hand_type == 'Right' \
                   else (lm[t[0]][0] < lm[t[0]-1][0])
        fingers = [1 if thumb_up else 0]
        fingers += [1 if lm[t[i]][1] < lm[t[i]-2][1] else 0 for i in range(1, 5)]
        return fingers

    def findDistance(self, p1, p2, img=None):
        """Відстань між двома точками (px). Повертає (length, img, info)."""
        x1, y1 = p1
        x2, y2 = p2
        cx, cy = (x1+x2)//2, (y1+y2)//2
        length = math.hypot(x2-x1, y2-y1)
        if img is not None:
            cv2.line(img, (x1,y1), (x2,y2), (255, 0, 255), 3)
            cv2.circle(img, (cx, cy), 8, (0, 255, 255), cv2.FILLED)
        return length, img, (x1, y1, x2, y2, cx, cy)

print('Бібліотеки імпортовано. HandDetector готовий до роботи.')


### 3. Ініціалізація HandDetector (mediapipe 0.10+)


In [ ]:
# detectionCon — поріг впевненості при виявленні руки (0.0–1.0)
# maxHands     — максимальна кількість рук для відстеження
detector = HandDetector(detectionCon=0.7, maxHands=1)

print("HandDetector ініціалізовано успішно")

### 4. Перевірка аудіо-пристроїв (PulseAudio)

In [ ]:
with pulsectl.Pulse('volume-check') as pulse:
    sinks = pulse.sink_list()
    print("Доступні пристрої виводу (динаміки):")
    for i, sink in enumerate(sinks):
        vol_pct = round(sink.volume.value_flat * 100)
        print(f"  [{i}] {sink.description} — гучність: {vol_pct}%")

### 5. Допоміжні функції керування гучністю

In [ ]:
def set_volume(level_pct: float):
    """Встановлює системну гучність. level_pct: 0–100"""
    level_pct = max(0.0, min(100.0, level_pct))
    with pulsectl.Pulse('set-volume') as pulse:
        sinks = pulse.sink_list()
        if sinks:
            pulse.volume_set_all_chans(sinks[0], level_pct / 100.0)

def get_volume() -> int:
    """Повертає поточну гучність у відсотках (0–100)"""
    with pulsectl.Pulse('get-volume') as pulse:
        sinks = pulse.sink_list()
        if sinks:
            return round(sinks[0].volume.value_flat * 100)
    return 0

print(f"Поточна гучність: {get_volume()}%")

### 6. Головний цикл — контроль гучності жестами

**Принцип роботи:**
- `lmList[4]` — кінчик великого пальця (landmark 4)
- `lmList[8]` — кінчик вказівного пальця (landmark 8)
- Відстань між ними (30–250 пікс.) → гучність (0–100%)

> Натисніть **`q`** для виходу

In [ ]:
cap = cv2.VideoCapture(1)

while True:
    success, img = cap.read()
    if not success:
        print("Не вдалося отримати зображення з камери")
        break

    # --- Розпізнавання рук (cvzone повертає список знайдених рук та зображення з розміткою) ---
    hands_found, img = detector.findHands(img)

    if hands_found:
        hand = hands_found[0]          # перша знайдена рука
        lmList = hand['lmList']        # список 21 landmark-а: [[x,y,z], ...]

        # Координати великого (4) та вказівного (8) пальців
        x1, y1 = lmList[4][0], lmList[4][1]
        x2, y2 = lmList[8][0], lmList[8][1]

        # Позначення пальців та лінії між ними
        cv2.circle(img, (x1, y1), 13, (255, 0, 0), cv2.FILLED)
        cv2.circle(img, (x2, y2), 13, (255, 0, 0), cv2.FILLED)
        cv2.line(img, (x1, y1), (x2, y2), (255, 0, 0), 3)

        # Відстань між пальцями через вбудований метод cvzone
        length, img, _ = detector.findDistance((x1, y1), (x2, y2), img)

        # Інтерполяція: 30–250 пікс. → 0–100%
        volper = float(np.interp(length, [30, 250], [0, 100]))
        volbar = int(np.interp(length,   [30, 250], [400, 150]))

        set_volume(volper)
        print(f"Гучність: {int(volper)}% | Відстань: {int(length)} пікс", end='\r')

        # --- Шкала гучності ---
        cv2.rectangle(img, (50, 150), (85, 400), (0, 0, 255), 4)
        cv2.rectangle(img, (50, volbar), (85, 400), (0, 0, 255), cv2.FILLED)
        cv2.putText(img, f"{int(volper)}%", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 98), 3)

    cv2.imshow('Volume Control', img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("\nПрограму завершено")

## Завдання 2 — Увімкнення/вимкнення мікрофону жестами

**Жест-перемикач:** стиснутий кулак (0 піднятих пальців) вмикає/вимикає мікрофон.
Метод `fingersUp()` з cvzone повертає список `[1/0, 1/0, 1/0, 1/0, 1/0]` для кожного пальця.

In [ ]:
import time

def set_mic_mute(muted: bool):
    """Вмикає або вимикає мікрофон через PulseAudio"""
    with pulsectl.Pulse('mic-control') as pulse:
        # Беремо перший реальний мікрофон (виключаємо monitor-пристрої)
        sources = [s for s in pulse.source_list() if 'monitor' not in s.name]
        if sources:
            pulse.mute(sources[0], muted)

def get_mic_mute() -> bool:
    """Повертає True якщо мікрофон вимкнено"""
    with pulsectl.Pulse('mic-status') as pulse:
        sources = [s for s in pulse.source_list() if 'monitor' not in s.name]
        if sources:
            return bool(sources[0].mute)
    return False

print(f"Поточний стан мікрофону: {'ВИМКНЕНО' if get_mic_mute() else 'УВІМКНЕНО'}")

In [ ]:
mic_muted = get_mic_mute()
last_toggle_time = 0
DEBOUNCE_SEC = 1.5   # затримка між перемиканнями (щоб уникнути випадкових спрацювань)

cap = cv2.VideoCapture(0)

while True:
    success, img = cap.read()
    if not success:
        break

    hands_found, img = detector.findHands(img)

    if hands_found:
        hand = hands_found[0]

        # fingersUp() повертає [великий, вказівний, середній, безіменний, мізинець]
        # 1 = піднятий, 0 = опущений
        fingers = detector.fingersUp(hand)
        fingers_up_count = sum(fingers)
        current_time = time.time()

        # Кулак (всі 0) + дебаунс → перемикаємо мікрофон
        if fingers_up_count == 0 and (current_time - last_toggle_time) > DEBOUNCE_SEC:
            mic_muted = not mic_muted
            set_mic_mute(mic_muted)
            last_toggle_time = current_time
            print(f"Мікрофон: {'ВИМКНЕНО' if mic_muted else 'УВІМКНЕНО'}")

        # Відображення стану на екрані
        mic_text  = "MIC: OFF" if mic_muted else "MIC: ON"
        mic_color = (0, 0, 255) if mic_muted else (0, 255, 0)
        cv2.putText(img, mic_text,  (10, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, mic_color, 3)
        cv2.putText(img, f"Fingers: {fingers_up_count}", (10, 130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    cv2.imshow('Microphone Control', img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Програму завершено")